# EmbdGuard Validation on MovieLens-1M

End-to-end validation of all EmbdGuard components on the full MovieLens-1M DLAttack pipeline:

1. **Detectors** — GradientAnomaly, AccessFrequency, EmbeddingDrift, GradientDistribution, TemporalAccess
2. **Defenses** — GradientClip, RowFreeze, InteractionFilter
3. **Evaluation Framework** — EvalRun harness, detector comparison
4. **Full DLAttack Pipeline** — Multi-round attack with EmbdGuard monitoring

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DLATTACK_DIR = os.path.join(REPO_ROOT, "dlattack_research")

# Import EmbdGuard modules
sys.path.insert(0, REPO_ROOT)

from src.guard import EmbdGuard
from src.detectors.gradient_anomaly import GradientAnomalyDetector
from src.detectors.access_frequency import AccessFrequencyDetector
from src.detectors.embedding_drift import EmbeddingDriftDetector
from src.detectors.gradient_distribution import GradientDistributionDetector
from src.detectors.temporal_access import TemporalAccessDetector
from src.defenses.gradient_clip import GradientClipDefense
from src.defenses.row_freeze import RowFreezeDefense
from src.defenses.interaction_filter import InteractionFilterDefense
from src.evaluation.harness import EvalRun, DataConfig, AttackConfig
from src.evaluation.compare import compare, format_comparison

import src as embdguard_src

# Import dlattack modules
sys.path.insert(0, DLATTACK_DIR)
del sys.modules["src"]
for key in list(sys.modules.keys()):
    if key.startswith("src."):
        del sys.modules[key]

import src.dataset as dl_dataset
import src.model as dl_model
import src.train as dl_train
import src.attack as dl_attack
import src.evaluate as dl_evaluate

# Restore EmbdGuard src
sys.modules["src"] = embdguard_src
sys.path.remove(DLATTACK_DIR)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cpu")
print(f"Repo: {REPO_ROOT}")
print(f"Device: {device}")

---
## 1. Load MovieLens-1M

In [ ]:
os.chdir(DLATTACK_DIR)
dl_dataset.download_ml1m()
df, n_users, n_items, user_map, item_map = dl_dataset.load_ratings()
train_df, test_df = dl_dataset.split_data(df)

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df):,} interactions")
print(f"Test:  {len(test_df):,} interactions")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Sample:\n{train_df.head()}")

In [ ]:
# Pick a target item (mid-popularity)
counts = train_df["item_id"].value_counts()
mid = counts[(counts > 20) & (counts < 100)].index.tolist()
TARGET = int(mid[0]) if mid else int(train_df["item_id"].mode()[0])
print(f"Target item: {TARGET} (count={counts[TARGET]})")

fig, ax = plt.subplots(1, 1, figsize=(10, 3))
ax.hist(counts.values, bins=100, edgecolor="black", alpha=0.7)
ax.axvline(counts[TARGET], color="red", linestyle="--", label=f"Target {TARGET} (n={counts[TARGET]})")
ax.set_xlabel("Interaction count")
ax.set_ylabel("Number of items")
ax.set_title("Item popularity distribution")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Load Baseline Model

In [ ]:
EMBED_DIM = 64
LAYER_SIZES = [128, 64]
LR = 0.001

ebc = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
two_tower = dl_model.TwoTower(ebc, layer_sizes=LAYER_SIZES, device=device)
model = dl_model.TwoTowerTrainTask(two_tower)

ckpt_path = os.path.join(DLATTACK_DIR, "checkpoints", "baseline.pt")
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state, strict=False)
    print(f"Loaded baseline from {ckpt_path}")
else:
    print(f"No checkpoint at {ckpt_path}, using random init")

optimizer = dl_model.make_optimizer(model, lr=LR)

baseline = dl_evaluate.evaluate(
    model.two_tower, test_df, train_df, n_items, n_neg=99, k=10, device=str(device)
)
print(f"\nBaseline HR@10 = {baseline['HR@K']:.4f}")
print(f"Baseline NDCG@10 = {baseline['NDCG@K']:.4f}")

---
## 3. Validate Evaluation Framework (Synthetic)

Run the built-in evaluation harness on synthetic data to confirm detectors work correctly in controlled conditions before moving to MovieLens-1M.

In [ ]:
configs = [
    {"name": "gradient_anomaly", "detectors": [GradientAnomalyDetector(threshold_z=3.0, min_steps=20)]},
    {"name": "access_frequency", "detectors": [AccessFrequencyDetector(concentration_threshold=5.0, min_steps=10)]},
    {"name": "embedding_drift", "detectors": [EmbeddingDriftDetector(drift_threshold_z=3.0, min_steps=10)]},
    {"name": "gradient_distribution", "detectors": [GradientDistributionDetector(kurtosis_z=3.0, concentration_threshold=10.0, min_steps=20)]},
    {"name": "temporal_access", "detectors": [TemporalAccessDetector(burst_window=5, burst_threshold=0.8, min_steps=10)]},
    {"name": "all_combined", "detectors": [
        AccessFrequencyDetector(concentration_threshold=5.0, min_steps=10),
        EmbeddingDriftDetector(drift_threshold_z=3.0, min_steps=10),
        TemporalAccessDetector(burst_window=5, burst_threshold=0.8, min_steps=10),
    ]},
]

results = compare(configs, data_config=DataConfig(), attack_config=AttackConfig())
print(format_comparison(results))

In [ ]:
# Visualize synthetic evaluation
names = [r["config"] for r in results]
precisions = [r["precision"] for r in results]
recalls = [r["recall"] for r in results]
f1s = [r["f1"] for r in results]
latencies = [r["detection_latency"] if r["detection_latency"] is not None else 0 for r in results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.arange(len(names))
w = 0.35
axes[0].bar(x - w/2, precisions, w, label="Precision", color="steelblue")
axes[0].bar(x + w/2, recalls, w, label="Recall", color="coral")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Score")
axes[0].set_title("Precision / Recall (Synthetic)")
axes[0].legend()
axes[0].set_ylim(0, 1.1)

axes[1].bar(x, f1s, color="seagreen")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 Score (Synthetic)")
axes[1].set_ylim(0, 1.1)

axes[2].bar(x, latencies, color="goldenrod")
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Steps")
axes[2].set_title("Detection Latency (Synthetic)")

plt.tight_layout()
plt.show()

---
## 4. Validate Defenses (Synthetic)

Verify each defense mechanism works correctly before using them in the ML-1M pipeline.

In [ ]:
# --- GradientClipDefense ---
print("=== GradientClipDefense ===")
test_ebc = dl_model.build_ebc(50, 100, 16, device=device)
test_tt = dl_model.TwoTower(test_ebc, layer_sizes=[32, 16], device=device)
test_model = dl_model.TwoTowerTrainTask(test_tt)

clip_def = GradientClipDefense(max_norm=0.001)
clip_def.apply(test_model)
clip_def.activate("t_item_id", [42], duration=5)

kjt = dl_model.make_kjt(torch.tensor([0, 1]), torch.tensor([42, 10]))
labels = torch.tensor([1.0, 0.0])
loss, _ = test_model(kjt, labels)
loss.backward()

weight = test_model.two_tower.ebc.embedding_bags["t_item_id"].weight
if weight.grad is not None:
    clipped_norm = weight.grad[42].norm().item()
    unclipped_norm = weight.grad[10].norm().item()
    print(f"  Row 42 (clipped) grad norm:   {clipped_norm:.6f} (max_norm=0.001)")
    print(f"  Row 10 (unclipped) grad norm: {unclipped_norm:.6f}")
    assert clipped_norm <= 0.001 + 1e-6, "Clip defense failed!"
    print("  PASSED")
clip_def.remove()

# --- RowFreezeDefense ---
print("\n=== RowFreezeDefense ===")
test_ebc = dl_model.build_ebc(50, 100, 16, device=device)
test_tt = dl_model.TwoTower(test_ebc, layer_sizes=[32, 16], device=device)
test_model = dl_model.TwoTowerTrainTask(test_tt)

freeze_def = RowFreezeDefense()
freeze_def.apply(test_model)
freeze_def.activate("t_item_id", [42], duration=5)

kjt = dl_model.make_kjt(torch.tensor([0, 1]), torch.tensor([42, 10]))
labels = torch.tensor([1.0, 0.0])
loss, _ = test_model(kjt, labels)
loss.backward()

weight = test_model.two_tower.ebc.embedding_bags["t_item_id"].weight
if weight.grad is not None:
    frozen_norm = weight.grad[42].norm().item()
    print(f"  Row 42 (frozen) grad norm: {frozen_norm:.10f}")
    assert frozen_norm < 1e-10, "Freeze defense failed!"
    print("  PASSED")
freeze_def.remove()

# --- InteractionFilterDefense ---
print("\n=== InteractionFilterDefense ===")
filter_def = InteractionFilterDefense()
filter_def.activate("t_item_id", [42, 99], duration=5)

users = torch.tensor([0, 1, 2, 3, 4])
items = torch.tensor([10, 42, 20, 99, 30])
labels = torch.tensor([1.0, 1.0, 0.0, 1.0, 0.0])

fu, fi, fl = filter_def.filter_batch(users, items, labels)
print(f"  Before filter: {len(users)} interactions, items={items.tolist()}")
print(f"  After filter:  {len(fu)} interactions, items={fi.tolist()}")
assert 42 not in fi.tolist() and 99 not in fi.tolist(), "Filter defense failed!"
print("  PASSED")
filter_def.remove()

# --- Defense expiration ---
print("\n=== Defense Expiration ===")
freeze_def = RowFreezeDefense()
freeze_def.activate("t_item_id", [42], duration=3)
for i in range(4):
    active = 42 in freeze_def.active_rows.get("t_item_id", [])
    print(f"  Step {i}: row 42 active = {active}")
    freeze_def.step()
print("  PASSED (expired after 3 steps)")

---
## 5. DLAttack with EmbdGuard Monitoring on MovieLens-1M

Run a multi-round DLAttack with all detectors attached. Track alerts per round and per detector.

In [ ]:
# Attack config
N_ROUNDS = 3
M_FAKE_USERS = 50
RETRAIN_EPOCHS = 5
BATCH_SIZE = 2048

# Reload baseline
ebc = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
two_tower = dl_model.TwoTower(ebc, layer_sizes=LAYER_SIZES, device=device)
model = dl_model.TwoTowerTrainTask(two_tower)
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state, strict=False)
optimizer = dl_model.make_optimizer(model, lr=LR)

eval_fn = lambda m: dl_evaluate.evaluate(
    m, test_df, train_df, n_items, n_neg=99, k=10, device=str(device)
)

print(f"Config: rounds={N_ROUNDS}, m={M_FAKE_USERS}, retrain_epochs={RETRAIN_EPOCHS}")
print(f"Target item: {TARGET}")

In [ ]:
poisoned_train = train_df.copy()
fake_user_id_start = n_users

# Storage for analysis
round_metrics = []  # (round, HR@10, NDCG@10, target_HR@10)
all_alerts_by_round = {}  # round -> list[Alert]
all_alerts_by_detector = defaultdict(list)  # detector_name -> list[Alert]
loss_history = []  # (round, epoch, loss)
alerts_per_epoch = []  # (round, epoch, n_alerts)

# Snapshot target item embedding before attack
target_emb_before = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data[TARGET].clone()

for r in range(1, N_ROUNDS + 1):
    print(f"\n{'='*60}")
    print(f"  ROUND {r}/{N_ROUNDS}")
    print(f"{'='*60}")

    # 1. Build surrogate
    max_uid = int(poisoned_train["user_id"].max()) + 1
    surrogate = dl_attack._build_surrogate(
        model, max_uid, n_items, EMBED_DIM, LAYER_SIZES, str(device)
    )
    surr_task = dl_model.TwoTowerTrainTask(surrogate)
    surr_opt = dl_model.make_optimizer(surr_task, lr=LR)
    print(f"  Retraining surrogate...")
    dl_train.train(surr_task, surr_opt, poisoned_train, n_items,
                   epochs=RETRAIN_EPOCHS, batch_size=BATCH_SIZE,
                   device=str(device), eval_fn=None, save_path=None)
    surrogate = surr_task.two_tower

    # 2. Generate fake users
    n_prev_fake = (r - 1) * M_FAKE_USERS
    print(f"  Generating {M_FAKE_USERS} fake users...")
    fake_df = dl_attack.generate_fake_users(
        surrogate, TARGET, n_users=n_users + n_prev_fake,
        n_items=n_items, m=M_FAKE_USERS, n_filler=30, n_optim_steps=200,
        fake_user_id_start=fake_user_id_start, device=str(device),
    )
    fake_user_id_start += M_FAKE_USERS

    # 3. Inject
    poisoned_train = pd.concat([poisoned_train, fake_df]).reset_index(drop=True)
    print(f"  Poisoned set: {len(poisoned_train):,} interactions")

    # 4. Build expanded model
    max_uid = int(poisoned_train["user_id"].max()) + 1
    new_tt = dl_attack._build_plain_two_tower(
        max_uid, n_items, EMBED_DIM, LAYER_SIZES, str(device)
    )
    dl_attack._copy_weights_to_plain(model, new_tt)
    model = dl_model.TwoTowerTrainTask(new_tt)
    optimizer = dl_model.make_optimizer(model, lr=LR)

    # 5. Retrain with EmbdGuard
    print(f"  Retraining with EmbdGuard...")
    guard = EmbdGuard(model)
    guard.add_detector(GradientAnomalyDetector(threshold_z=5.0, min_steps=50))
    guard.add_detector(AccessFrequencyDetector(concentration_threshold=3.0, min_steps=30))
    guard.add_detector(EmbeddingDriftDetector(
        drift_threshold_z=8.0, min_steps=50, snapshot_interval=100))
    guard.add_detector(GradientDistributionDetector(
        kurtosis_z=5.0, concentration_threshold=50.0, min_steps=50))
    guard.add_detector(TemporalAccessDetector(
        burst_window=10, burst_threshold=1.0, top_k=3, min_steps=50))

    pos_users = torch.tensor(poisoned_train["user_id"].values, dtype=torch.long, device=device)
    pos_items = torch.tensor(poisoned_train["item_id"].values, dtype=torch.long, device=device)

    round_alerts = []
    step = 0
    for epoch in range(1, RETRAIN_EPOCHS + 1):
        model.train()
        users, items, labels = dl_train._negative_sample_tensors(
            pos_users, pos_items, n_items, 4, device
        )
        n_samples = len(users)
        total_loss = 0.0
        epoch_alerts = 0

        for start in range(0, n_samples, BATCH_SIZE):
            end = min(start + BATCH_SIZE, n_samples)
            kjt = dl_model.make_kjt(users[start:end], items[start:end])
            batch_labels = labels[start:end]

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                loss, _ = model(kjt, batch_labels)
                optimizer.zero_grad()
                loss.backward()
            optimizer.step()

            step += 1
            alerts = guard.step()
            round_alerts.extend(alerts)
            epoch_alerts += len(alerts)
            total_loss += loss.item() * (end - start)

        train_loss = total_loss / n_samples
        loss_history.append((r, epoch, train_loss))
        alerts_per_epoch.append((r, epoch, epoch_alerts))
        print(f"    Epoch {epoch:2d} | loss={train_loss:.4f} | alerts={epoch_alerts}")

    guard.detach()

    # Store alerts
    all_alerts_by_round[r] = round_alerts
    for a in round_alerts:
        all_alerts_by_detector[a.detector].append(a)

    # Evaluate
    metrics = eval_fn(model.two_tower)
    target_hr = dl_evaluate.target_item_hit_ratio(
        model.two_tower, TARGET, test_df, train_df, n_items,
        n_neg=99, k=10, device=str(device)
    )
    round_metrics.append((r, metrics["HR@K"], metrics["NDCG@K"], target_hr))
    print(f"\n  HR@10={metrics['HR@K']:.4f} | NDCG@10={metrics['NDCG@K']:.4f} | Target HR@10={target_hr:.4f}")

# Snapshot target embedding after attack
target_emb_after = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data[TARGET].clone()

---
## 6. Analysis: Alert Distribution by Detector

In [ ]:
# Summary table
print(f"{'Detector':<25} {'Alerts':>8} {'First Step':>12}")
print("-" * 48)
for det_name in sorted(all_alerts_by_detector.keys()):
    alerts = all_alerts_by_detector[det_name]
    first = min(a.step for a in alerts)
    print(f"{det_name:<25} {len(alerts):>8,} {first:>12}")
total = sum(len(a) for a in all_alerts_by_round.values())
print(f"{'TOTAL':<25} {total:>8,}")

In [ ]:
# Alert counts by detector (exclude embedding_drift for readability)
det_names = sorted(all_alerts_by_detector.keys())
det_counts = {d: len(all_alerts_by_detector[d]) for d in det_names}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# All detectors
axes[0].barh(det_names, [det_counts[d] for d in det_names], color="steelblue")
axes[0].set_xlabel("Alert count")
axes[0].set_title("Alerts by Detector (all)")
axes[0].set_xscale("log")

# Excluding embedding_drift to see the rest clearly
other = {d: c for d, c in det_counts.items() if d != "embedding_drift"}
axes[1].barh(list(other.keys()), list(other.values()), color="coral")
axes[1].set_xlabel("Alert count")
axes[1].set_title("Alerts by Detector (excl. embedding_drift)")

plt.tight_layout()
plt.show()

In [ ]:
# Alerts per epoch across rounds
fig, ax = plt.subplots(figsize=(10, 4))
epochs_flat = list(range(len(alerts_per_epoch)))
alert_vals = [x[2] for x in alerts_per_epoch]
colors = [f"C{x[0]-1}" for x in alerts_per_epoch]

ax.bar(epochs_flat, alert_vals, color=colors, edgecolor="black", alpha=0.7)
ax.set_xlabel("Epoch (across rounds)")
ax.set_ylabel("Alerts")
ax.set_title("Alert Count per Epoch")

# Add round separators
for i, (r, e, _) in enumerate(alerts_per_epoch):
    if e == 1 and i > 0:
        ax.axvline(i - 0.5, color="gray", linestyle="--", alpha=0.5)
    if e == 1:
        ax.text(i + RETRAIN_EPOCHS / 2 - 0.5, max(alert_vals) * 0.95,
                f"Round {r}", ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Analysis: Model Quality Over Rounds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

rounds = [m[0] for m in round_metrics]
hr = [m[1] for m in round_metrics]
ndcg = [m[2] for m in round_metrics]
thr = [m[3] for m in round_metrics]

# HR@10
axes[0].plot(rounds, hr, "o-", color="steelblue", linewidth=2)
axes[0].axhline(baseline["HR@K"], color="gray", linestyle="--", label="Baseline")
axes[0].set_xlabel("Attack Round")
axes[0].set_ylabel("HR@10")
axes[0].set_title("Overall Recommendation Quality")
axes[0].legend()

# NDCG@10
axes[1].plot(rounds, ndcg, "o-", color="coral", linewidth=2)
axes[1].axhline(baseline["NDCG@K"], color="gray", linestyle="--", label="Baseline")
axes[1].set_xlabel("Attack Round")
axes[1].set_ylabel("NDCG@10")
axes[1].set_title("Ranking Quality")
axes[1].legend()

# Target HR@10
axes[2].plot(rounds, thr, "o-", color="red", linewidth=2, markersize=8)
axes[2].set_xlabel("Attack Round")
axes[2].set_ylabel("Target HR@10")
axes[2].set_title(f"Target Item {TARGET} Promotion")
axes[2].set_ylim(-0.01, max(0.05, max(thr) * 1.2))

plt.tight_layout()
plt.show()

In [ ]:
# Training loss across epochs
fig, ax = plt.subplots(figsize=(10, 3))
epochs_flat = list(range(len(loss_history)))
losses = [x[2] for x in loss_history]
colors = [f"C{x[0]-1}" for x in loss_history]

ax.plot(epochs_flat, losses, "o-", color="steelblue", markersize=4)
for i, (r, e, _) in enumerate(loss_history):
    if e == 1 and i > 0:
        ax.axvline(i - 0.5, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Epoch (across rounds)")
ax.set_ylabel("Train Loss")
ax.set_title("Training Loss")
plt.tight_layout()
plt.show()

---
## 8. Analysis: Target Item Embedding Drift

In [ ]:
drift = (target_emb_after - target_emb_before).norm().item()
print(f"Target item {TARGET} embedding L2 drift: {drift:.4f}")

# Compare with random items
item_weights = model.two_tower.ebc.embedding_bags["t_item_id"].weight.data
ebc_before = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
tt_before = dl_model.TwoTower(ebc_before, layer_sizes=LAYER_SIZES, device=device)
m_before = dl_model.TwoTowerTrainTask(tt_before)
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    m_before.load_state_dict(state, strict=False)
weights_before = m_before.two_tower.ebc.embedding_bags["t_item_id"].weight.data

n_sample_items = min(500, n_items)
sample_ids = np.random.choice(n_items, n_sample_items, replace=False)
drifts = []
for sid in sample_ids:
    d = (item_weights[sid] - weights_before[sid]).norm().item()
    drifts.append(d)

target_drift = (item_weights[TARGET] - weights_before[TARGET]).norm().item()

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(drifts, bins=50, alpha=0.7, color="steelblue", edgecolor="black", label="Random items")
ax.axvline(target_drift, color="red", linestyle="--", linewidth=2,
           label=f"Target {TARGET} (drift={target_drift:.3f})")
ax.set_xlabel("L2 drift from baseline")
ax.set_ylabel("Count")
ax.set_title("Embedding Drift Distribution (Baseline vs Post-Attack)")
ax.legend()
plt.tight_layout()
plt.show()

# Percentile of target item
pct = np.mean([d <= target_drift for d in drifts]) * 100
print(f"Target item drift is at {pct:.1f}th percentile among sampled items")

---
## 9. Analysis: Which Items Get Flagged?

In [ ]:
# Check access_frequency flagged rows
for det_name in ["access_frequency", "temporal_access"]:
    if det_name in all_alerts_by_detector:
        flagged_rows = set()
        for a in all_alerts_by_detector[det_name]:
            row = a.details.get("hottest_row") or a.details.get("row_id")
            if row is not None:
                flagged_rows.add(int(row))
        print(f"{det_name}: flagged {len(flagged_rows)} unique rows")
        print(f"  Rows: {sorted(flagged_rows)[:20]}")
        if TARGET in flagged_rows:
            print(f"  Target {TARGET} WAS flagged")
        else:
            print(f"  Target {TARGET} was NOT flagged")

        # Check overlap with popular items
        top_popular = counts.head(20).index.tolist()
        overlap = flagged_rows & set(top_popular)
        print(f"  Overlap with top-20 popular items: {sorted(overlap)}")
        print()

In [ ]:
# Embedding drift: how many rows are flagged, and is target among them?
if "embedding_drift" in all_alerts_by_detector:
    drift_flagged = set()
    for a in all_alerts_by_detector["embedding_drift"]:
        row = a.details.get("row_id")
        if row is not None:
            drift_flagged.add(int(row))
    print(f"embedding_drift: flagged {len(drift_flagged)} unique rows out of {n_items} items")
    print(f"  Coverage: {len(drift_flagged)/n_items*100:.1f}%")
    if TARGET in drift_flagged:
        print(f"  Target {TARGET} WAS flagged (but precision is {1/len(drift_flagged):.6f})")
    else:
        print(f"  Target {TARGET} was NOT flagged")

---
## 10. Defense Impact: RowFreeze During Attack

Re-run one round of the attack with RowFreezeDefense active to measure its effect on attack success and model quality.

In [ ]:
# Reload baseline for fair comparison
ebc_def = dl_model.build_ebc(n_users, n_items, EMBED_DIM, device=device)
tt_def = dl_model.TwoTower(ebc_def, layer_sizes=LAYER_SIZES, device=device)
model_def = dl_model.TwoTowerTrainTask(tt_def)
if os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_def.load_state_dict(state, strict=False)
opt_def = dl_model.make_optimizer(model_def, lr=LR)

# Build surrogate, generate fakes
max_uid = int(train_df["user_id"].max()) + 1
surr = dl_attack._build_surrogate(model_def, max_uid, n_items, EMBED_DIM, LAYER_SIZES, str(device))
surr_t = dl_model.TwoTowerTrainTask(surr)
surr_o = dl_model.make_optimizer(surr_t, lr=LR)
print("Retraining surrogate...")
dl_train.train(surr_t, surr_o, train_df, n_items, epochs=RETRAIN_EPOCHS,
               batch_size=BATCH_SIZE, device=str(device), eval_fn=None, save_path=None)

print(f"Generating {M_FAKE_USERS} fake users...")
fake_df = dl_attack.generate_fake_users(
    surr_t.two_tower, TARGET, n_users=n_users,
    n_items=n_items, m=M_FAKE_USERS, n_filler=30, n_optim_steps=200,
    fake_user_id_start=n_users, device=str(device)
)

poisoned_def = pd.concat([train_df, fake_df]).reset_index(drop=True)
max_uid = int(poisoned_def["user_id"].max()) + 1
new_tt = dl_attack._build_plain_two_tower(max_uid, n_items, EMBED_DIM, LAYER_SIZES, str(device))
dl_attack._copy_weights_to_plain(model_def, new_tt)
model_def = dl_model.TwoTowerTrainTask(new_tt)
opt_def = dl_model.make_optimizer(model_def, lr=LR)

print(f"Poisoned set: {len(poisoned_def):,} interactions")

In [ ]:
# Retrain WITH RowFreezeDefense
print("Retraining with EmbdGuard + RowFreezeDefense...")
guard_def = EmbdGuard(model_def)
guard_def.add_detector(AccessFrequencyDetector(concentration_threshold=3.0, min_steps=30))
guard_def.add_detector(TemporalAccessDetector(burst_window=10, burst_threshold=1.0, top_k=3, min_steps=50))
guard_def.add_defense(RowFreezeDefense())

pos_u = torch.tensor(poisoned_def["user_id"].values, dtype=torch.long, device=device)
pos_i = torch.tensor(poisoned_def["item_id"].values, dtype=torch.long, device=device)

defense_alerts = []
defense_losses = []
frozen_rows_per_epoch = []

step = 0
for epoch in range(1, RETRAIN_EPOCHS + 1):
    model_def.train()
    users, items, labels = dl_train._negative_sample_tensors(pos_u, pos_i, n_items, 4, device)
    n_samples = len(users)
    total_loss = 0.0
    epoch_alerts = 0

    for start in range(0, n_samples, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_samples)
        kjt = dl_model.make_kjt(users[start:end], items[start:end])
        batch_labels = labels[start:end]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            loss, _ = model_def(kjt, batch_labels)
            opt_def.zero_grad()
            loss.backward()
        opt_def.step()

        step += 1
        alerts = guard_def.step()
        defense_alerts.extend(alerts)
        epoch_alerts += len(alerts)
        total_loss += loss.item() * (end - start)

    train_loss = total_loss / n_samples
    defense_losses.append(train_loss)

    # Count frozen rows
    n_frozen = sum(len(rows) for rows in guard_def._defenses[0].active_rows.values()) if guard_def._defenses else 0
    frozen_rows_per_epoch.append(n_frozen)
    print(f"  Epoch {epoch:2d} | loss={train_loss:.4f} | alerts={epoch_alerts} | frozen_rows={n_frozen}")

guard_def.detach()

# Evaluate
met_def = eval_fn(model_def.two_tower)
thr_def = dl_evaluate.target_item_hit_ratio(
    model_def.two_tower, TARGET, test_df, train_df, n_items,
    n_neg=99, k=10, device=str(device)
)
print(f"\nWith defense: HR@10={met_def['HR@K']:.4f} | NDCG@10={met_def['NDCG@K']:.4f} | Target HR@10={thr_def:.4f}")
print(f"Without defense (Round 1): HR@10={round_metrics[0][1]:.4f} | NDCG@10={round_metrics[0][2]:.4f} | Target HR@10={round_metrics[0][3]:.4f}")

In [ ]:
# Frozen rows over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, RETRAIN_EPOCHS + 1), frozen_rows_per_epoch, color="indianred", edgecolor="black")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Frozen rows")
axes[0].set_title("Active RowFreeze Defenses")

axes[1].plot(range(1, RETRAIN_EPOCHS + 1), defense_losses, "o-", color="steelblue", label="With defense")
r1_losses = [x[2] for x in loss_history if x[0] == 1]
axes[1].plot(range(1, RETRAIN_EPOCHS + 1), r1_losses[:RETRAIN_EPOCHS], "o--", color="coral", label="Without defense")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Training Loss: Defense vs No Defense")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 11. Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {"Condition": "Baseline (no attack)",
     "HR@10": baseline["HR@K"], "NDCG@10": baseline["NDCG@K"],
     "Target HR@10": 0.0, "Total Alerts": 0},
    {"Condition": f"After {N_ROUNDS} rounds (no defense)",
     "HR@10": round_metrics[-1][1], "NDCG@10": round_metrics[-1][2],
     "Target HR@10": round_metrics[-1][3],
     "Total Alerts": sum(len(a) for a in all_alerts_by_round.values())},
    {"Condition": "1 round + RowFreeze defense",
     "HR@10": met_def["HR@K"], "NDCG@10": met_def["NDCG@K"],
     "Target HR@10": thr_def,
     "Total Alerts": len(defense_alerts)},
])
print(comparison.to_string(index=False))

---
## 12. Key Findings

### Detector Validation
- All 5 detectors fire during poisoned retraining on MovieLens-1M
- **Synthetic data**: Perfect or near-perfect precision/recall (see Section 3)
- **MovieLens-1M**: High alert volume due to scale mismatch — per-step detectors detect normal training dynamics as anomalous at this scale

### Defense Validation
- GradientClipDefense correctly clips flagged row gradients to max_norm
- RowFreezeDefense zeroes gradients on flagged rows completely
- InteractionFilterDefense removes flagged item interactions from batches
- All defenses correctly auto-expire after configured duration

### Scale Gap
- DLAttack with 50 fake users/round (1,550 interactions/round) is <0.2% of the ~1M real interactions — insufficient to move Target HR@10 above 0
- Per-step embedding drift detector flags too many rows at real-data scale (normal training causes widespread drift)
- Access frequency detector correctly identifies naturally popular items but cannot distinguish attack targets

### Implications
- EmbdGuard's detectors and defenses are **functionally correct** (validated on synthetic + real data)
- Real-world detection requires **cross-epoch** or **user-level** analysis rather than per-step z-scores
- The DLAttack itself needs a higher poisoning ratio to be effective at MovieLens-1M scale